In [11]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/dmitryvolovikov/cram-school-easter-round-1-task/train_data.csv
/kaggle/input/datasets/dmitryvolovikov/cram-school-easter-round-1-task/test_data.csv
/kaggle/input/datasets/dmitryvolovikov/cram-school-easter-round-1-task/sample_output.csv


In [12]:
train_ds = pd.read_csv("/kaggle/input/datasets/dmitryvolovikov/cram-school-easter-round-1-task/train_data.csv")
test_ds = pd.read_csv("/kaggle/input/datasets/dmitryvolovikov/cram-school-easter-round-1-task/test_data.csv")
sample_sub = pd.read_csv("/kaggle/input/datasets/dmitryvolovikov/cram-school-easter-round-1-task/sample_output.csv")
import matplotlib.pyplot as plt
import seaborn as sns

In [40]:
#subtask 1
am_or_pm = pd.to_datetime(test_ds["Timestamp"]).dt.strftime("%p")
test_ds["am_or_pm"] = am_or_pm
train_ds["am_or_pm"] = pd.to_datetime(train_ds["Timestamp"]).dt.strftime("%p")
sample_sub.loc[sample_sub["subtaskID"] ==1, "answer"] = am_or_pm

In [14]:
train_ds.describe()

,ID,Suspicious_Port_Activity,Traffic_Volume_Variation,Packet_Length_Anomaly,Malware_Score,Threat_Level_Index,User_Behavior_Score,Geo_Dispersion,Payload_Entropy,Login_Attempts,Device_Response_Time,Session_Duration,Packet_Retry_Rate,Anomaly_Tendency,Attack Type
count,13356.000000,13356.000000,13356.000000,13356.000000,13356.000000,13356.000000,12057.000000,13356.000000,13356.000000,1.335600e+04,1.335600e+04,13356.000000,13356.000000,13356.000000,13356.000000
mean,19904.823600,49.791904,49.788837,49.807553,7.223436,4.070842,-0.001211,50.172048,11.148770,1.407737e+05,4.828117e+06,15.143093,19.936004,1.604717,0.745283
std,11550.807566,10.075306,10.080638,10.209363,14.450361,6.802323,1.008993,28.621882,30.631642,8.748305e+07,7.278543e+06,13.130384,6.446938,15.255836,0.827976
min,7.000000,9.830626,11.447170,9.704399,-40.898692,-20.865728,-4.207929,0.002713,-102.528539,-8.428669e+08,0.000000e+00,-35.197389,-6.683795,-73.746635,0.000000
25%,9985.750000,43.117314,43.043988,42.978653,-2.796696,-0.800068,-0.679533,25.838683,-9.625675,3.686437e+00,0.000000e+00,5.967130,15.665370,-8.751905,0.000000
50%,19880.500000,49.912484,49.946217,49.918006,6.839157,3.734464,-0.005401,50.284962,11.211696,1.449021e+03,0.000000e+00,15.215888,19.921653,1.568656,0.000000
75%,29923.500000,56.573445,56.550146,56.620733,16.848907,8.794495,0.673218,74.787214,31.656160,2.913327e+05,7.785192e+06,24.254570,24.321041,11.893369,1.000000
max,39996.000000,97.241773,97.398056,98.353307,63.855348,27.272845,4.185208,99.989204,138.235090,8.818372e+08,3.999218e+07,65.088692,44.933708,58.459195,2.000000


In [15]:
train_ds.isnull().sum()

ID                             0
Timestamp                      0
Suspicious_Port_Activity       0
Traffic_Volume_Variation       0
Packet_Length_Anomaly          0
Malware_Score                  0
Threat_Level_Index             0
User_Behavior_Score         1299
Geo_Dispersion                 0
Payload_Entropy                0
Login_Attempts                 0
Device_Response_Time           0
Session_Duration               0
Packet_Retry_Rate              0
Anomaly_Tendency               0
Attack Type                    0
dtype: int64

In [16]:
train_ds["User_Behavior_Score"].describe()

count    12057.000000
mean        -0.001211
std          1.008993
min         -4.207929
25%         -0.679533
50%         -0.005401
75%          0.673218
max          4.185208
Name: User_Behavior_Score, dtype: float64

In [17]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer().fit(train_ds[["User_Behavior_Score"]])
train_ds[["User_Behavior_Score"]]= imputer.transform(train_ds[["User_Behavior_Score"]])
test_ds[["User_Behavior_Score"]] = imputer.transform(test_ds[['User_Behavior_Score']])


In [18]:
cat_cols = train_ds.select_dtypes(include = ['object' , "category"]).columns
num_cols= train_ds.select_dtypes(include =["number"]).columns
cat_cols

Index(['Timestamp'], dtype='object')

In [24]:
train_ds["Timestamp"] = pd.to_datetime(train_ds["Timestamp"])
test_ds["Timestamp"] = pd.to_datetime(test_ds["Timestamp"])

train_ds["hour"] =train_ds["Timestamp"].dt.hour
test_ds["hour"] = test_ds["Timestamp"].dt.hour
train_ds['DayOfWeek'] = train_ds['Timestamp'].dt.dayofweek
test_ds['DayOfWeek'] = test_ds['Timestamp'].dt.dayofweek

train_ds['hour_sin'] = np.sin(2 * np.pi * train_ds['hour'] / 24)
train_ds['hour_cos'] = np.cos(2 * np.pi * train_ds['hour'] / 24)
test_ds['hour_sin'] = np.sin(2 * np.pi * test_ds['hour'] / 24)
test_ds['hour_cos'] = np.cos(2 * np.pi * test_ds['hour'] / 24)

In [34]:
X_cols = [col for col in train_ds.columns if col not in ["Timestamp", "hour", "DayOfWeek", "Attack Type","ID"]]
X_cols.remove("am_or_pm")

In [35]:
train_ds[X_cols].describe()

,Suspicious_Port_Activity,Traffic_Volume_Variation,Packet_Length_Anomaly,Malware_Score,Threat_Level_Index,User_Behavior_Score,Geo_Dispersion,Payload_Entropy,Login_Attempts,Device_Response_Time,Session_Duration,Packet_Retry_Rate,Anomaly_Tendency,hour_sin,hour_cos
count,13356.000000,13356.000000,13356.000000,13356.000000,13356.000000,13356.000000,13356.000000,13356.000000,1.335600e+04,1.335600e+04,13356.000000,13356.000000,13356.000000,1.335600e+04,1.335600e+04
mean,49.791904,49.788837,49.807553,7.223436,4.070842,-0.001211,50.172048,11.148770,1.407737e+05,4.828117e+06,15.143093,19.936004,1.604717,1.184052e-02,-7.338696e-03
std,10.075306,10.080638,10.209363,14.450361,6.802323,0.958667,28.621882,30.631642,8.748305e+07,7.278543e+06,13.130384,6.446938,15.255836,7.056146e-01,7.085117e-01
min,9.830626,11.447170,9.704399,-40.898692,-20.865728,-4.207929,0.002713,-102.528539,-8.428669e+08,0.000000e+00,-35.197389,-6.683795,-73.746635,-1.000000e+00,-1.000000e+00
25%,43.117314,43.043988,42.978653,-2.796696,-0.800068,-0.597838,25.838683,-9.625675,3.686437e+00,0.000000e+00,5.967130,15.665370,-8.751905,-7.071068e-01,-7.071068e-01
50%,49.912484,49.946217,49.918006,6.839157,3.734464,-0.001211,50.284962,11.211696,1.449021e+03,0.000000e+00,15.215888,19.921653,1.568656,1.224647e-16,-1.836970e-16
75%,56.573445,56.550146,56.620733,16.848907,8.794495,0.592959,74.787214,31.656160,2.913327e+05,7.785192e+06,24.254570,24.321041,11.893369,7.071068e-01,7.071068e-01
max,97.241773,97.398056,98.353307,63.855348,27.272845,4.185208,99.989204,138.235090,8.818372e+08,3.999218e+07,65.088692,44.933708,58.459195,1.000000e+00,1.000000e+00


In [36]:
from sklearn.preprocessing import StandardScaler

scaler= StandardScaler().fit(train_ds[X_cols])
train_ds[X_cols] = scaler.transform(train_ds[X_cols])
test_ds[X_cols] = scaler.transform(test_ds[X_cols])

In [38]:
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators =1000, max_depth = 6).fit(train_ds[X_cols], train_ds["Attack Type"])

In [41]:
sample_sub.loc[sample_sub["subtaskID"] == 2, "answer"] = xgb.predict(test_ds[X_cols])

In [43]:
sample_sub.to_csv("sub1.csv", index= False)